<a href="https://colab.research.google.com/github/sifat-lab/SpikeSoil-ML_powered_renewable_energy/blob/main/ml/Copy_of_ml_03_soiling_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SpikeSoil — ml_03: Soiling Dataset Builder (Phase C, step 1)Turns raw 5-second dual-panel logs into a **labelled soiling dataset**.**Ground truth**  `loss = 1 - (i_B / i_A) / r0`, where `r0` is the clean-cleancurrent ratio measured in the same session.**Model inputs must never include panel A.** Panel A exists only to createlabels; the deployed node sees one panel. Feature columns below are panel-Band irradiance only.Sessions: 2026-07-28 (5 dust levels, 10-25 ml steps) and 2026-07-31(4 levels, 40 ml steps). Slurry: 5 g sieved soil in 500 ml water -> 0.01 g/ml.Panel area 21 x 15 cm = 0.0315 m^2.

In [ ]:
import pandas as pd, numpy as np

pd.set_option("display.width", 200)
AREA = 0.0315        # m^2
CONC = 0.01          # g per ml of slurry
LUX_MIN, IA_MIN = 5000, 20.0     # daylight + non-trivial current
SHADOW_CUTOFF = "14:30"          # rig is shadowed after this - see notes
SETTLE_MIN = 30                  # minutes discarded after each application

## 1. Session definitionsApplication times come from the field notebook. `clean` is the clean-cleanwindow used to measure `r0` for that session.

In [ ]:
SESSIONS = {
    "2026-07-28": {
        "clean": ("08:30", "09:30"),
        "apps": [("09:40", 10), ("10:36", 10), ("11:25", 10), ("12:17", 20), ("13:36", 25)],
        "end": "14:20"
    },
    "2026-07-31": {
        "clean": ("09:45", "09:58"),
        "apps": [("10:00", 40), ("10:45", 40), ("11:30", 40), ("12:32", 40)],
        "end": "14:12"
    },
}
# Correcting paths to match the available files in /content/
FILES = {
    "2026-07-28": "log_20260728.csv",
    "2026-07-31": "log_20260731.csv"
}

## 2. Load and filterThree fixed cleanups, each for a documented hardware reason:* **drop row 0** - DS18B20 reports 85.0 C on its first read after boot* **`lux > 5000`** - low light makes the ratio explode (dividing small by small)* **`iA > 20 mA`** - same guard on the denominator`pA_mW` / `pB_mW` from the firmware are too coarsely quantised; power isrecomputed as `V x I` in Python.

In [ ]:
def load(f):
    d = pd.read_csv(f)
    d["t"] = pd.to_datetime(d["timestamp"])
    d = d.iloc[1:]                                   # DS18B20 boot artefact
    d = d[(d.lux > LUX_MIN) & (d.iA_mA > IA_MIN)].copy()
    d["pA"] = d.vA * d.iA_mA / 1000.0
    d["pB"] = d.vB * d.iB_mA / 1000.0
    d["r"]  = d.iB_mA / d.iA_mA
    return d.set_index("t").sort_index()

## 3. Plateau detectionA slurry application wets the panel; the ratio drifts for tens of minuteswhile it dries, and the operator's hands/shadow spike it. Only the**settled plateau** carries a trustworthy label.Rule: discard `SETTLE_MIN` after each application, then keep the longestcontiguous run of 5-min medians staying within +/-2.5% of the run median(minimum 10 minutes). Blocks that never stabilise are **dropped, notsalvaged** - a wrong label is worse than a missing one.

In [ ]:
def stable_run(block, tol=0.025, min_minutes=10):
    b = block["r"].resample("5min").median().dropna()
    if len(b) < 2: return None
    best = None
    for i in range(len(b)):
        for j in range(len(b), i, -1):
            seg = b.iloc[i:j]
            if len(seg) * 5 < min_minutes: break
            if (seg - seg.median()).abs().max() <= tol:
                if best is None or len(seg) > best[0]:
                    best = (len(seg), seg.index[0],
                            seg.index[-1] + pd.Timedelta("5min"), seg.median())
                break
    if best is None: return None
    return best[1], min(best[2], block.index[-1]), best[3]

## 4. Build labelled rows

In [ ]:
rows, report = [], []
for day, cfg in SESSIONS.items():
    d = load(FILES[day])
    tz = d.index.tz

    c0, c1 = [f"{day} {x}" for x in cfg["clean"]]
    r0 = d.loc[c0:c1, "r"].median()
    r0_sd = d.loc[c0:c1, "r"].resample("5min").median().std()
    report.append(f"{day}: r0 = {r0:.4f}  (5-min sd {r0_sd:.4f})")

    blk = d.loc[c0:c1].copy()
    blk["gm2"], blk["level"], blk["r0"], blk["session"] = 0.0, 0, r0, day
    rows.append(blk)

    bounds = [f"{day} {a}" for a, _ in cfg["apps"]] + [f"{day} {cfg['end']}"]
    cum_ml = 0
    for i, (app, ml) in enumerate(cfg["apps"], start=1):
        cum_ml += ml
        gm2 = cum_ml * CONC / AREA
        s = pd.Timestamp(f"{day} {app}").tz_localize(tz) + pd.Timedelta(minutes=SETTLE_MIN)
        e = min(pd.Timestamp(bounds[i]).tz_localize(tz),
                pd.Timestamp(f"{day} {SHADOW_CUTOFF}").tz_localize(tz))
        blk = d.loc[s:e]
        st = stable_run(blk) if len(blk) else None
        if st is None:
            report.append(f"  L{i}  {gm2:5.1f} g/m2  DROPPED (never stabilised)")
            continue
        blk = d.loc[st[0]:st[1]].copy()
        blk["gm2"], blk["level"], blk["r0"], blk["session"] = gm2, i, r0, day
        report.append(f"  L{i}  {st[0]:%H:%M}-{st[1]:%H:%M}  {gm2:5.1f} g/m2  "
                      f"r={st[2]:.3f}  loss={100*(1-st[2]/r0):5.1f}%  n={len(blk)}")
        rows.append(blk)

df = pd.concat(rows)
df["loss"] = (1 - df["r"] / df["r0"]).clip(-0.05, 1.0)
print("\n".join(report))
print("\nlabelled rows:", len(df))
df.to_csv("soiling_rows.csv")

2026-07-28: r0 = 0.9986  (5-min sd 0.0046)
  L1  10:10-10:35    3.2 g/m2  r=0.955  loss=  4.3%  n=300
  L2  11:05-11:24    6.3 g/m2  r=0.860  loss= 13.8%  n=240
  L3  11:55-12:05    9.5 g/m2  r=0.791  loss= 20.8%  n=120
  L4  12:45-13:35   15.9 g/m2  r=0.706  loss= 29.3%  n=600
  L5  14:05-14:19   23.8 g/m2  r=0.590  loss= 40.9%  n=180
2026-07-31: r0 = 1.0045  (5-min sd 0.0090)
  L1   12.7 g/m2  DROPPED (never stabilised)
  L2  11:15-11:29   25.4 g/m2  r=0.602  loss= 40.0%  n=180
  L3  12:05-12:31   38.1 g/m2  r=0.494  loss= 50.8%  n=324
  L4  13:35-14:11   50.8 g/m2  r=0.452  loss= 55.0%  n=444

labelled rows: 3246


## 5. Window features — **panel B only**5-minute windows. Everything here is measurable by a single-panel node.`iB_per_klux` is the physically meaningful one: current normalised byirradiance is essentially a performance ratio, so the network does not haveto learn division from 53 examples.

In [ ]:
W = []
for (sess, lvl), g in df.groupby(["session", "level"]):
    for ts, w in g.sort_index().resample("5min"):
        if len(w) < 40: continue
        h = ts.hour + ts.minute / 60
        W.append(dict(session=sess, level=lvl, t_start=ts,
            vB_mean=w.vB.mean(), vB_std=w.vB.std(),
            iB_mean=w.iB_mA.mean(), iB_std=w.iB_mA.std(), pB_mean=w.pB.mean(),
            lux_mean=w.lux.mean(), lux_std=w.lux.std(),
            tB_mean=w.tB_C.mean(), tB_rise=w.tB_C.iloc[-1] - w.tB_C.iloc[0],
            iB_per_klux=1000 * w.iB_mA.mean() / max(w.lux.mean(), 1),
            pB_per_klux=1000 * w.pB.mean() / max(w.lux.mean(), 1),
            hour_sin=np.sin(2*np.pi*h/24), hour_cos=np.cos(2*np.pi*h/24),
            gm2=w.gm2.iloc[0], loss=w.loss.median(), n=len(w)))
win = pd.DataFrame(W).sort_values(["session", "t_start"])
win.to_csv("soiling_dataset_5min.csv", index=False)
print("windows:", len(win))
print(win.groupby(["session", "level"]).agg(n=("loss","size"), gm2=("gm2","first"),
    loss=("loss","mean"), lux_lo=("lux_mean","min"), lux_hi=("lux_mean","max")).round(3))

windows: 53
                   n     gm2   loss     lux_lo     lux_hi
session    level                                         
2026-07-28 0      12   0.000  0.001  13800.471  62810.649
           1       5   3.175  0.045  23848.897  67742.944
           2       4   6.349  0.139  37663.890  59647.530
           3       2   9.524  0.208  26225.347  27367.031
           4      10  15.873  0.293  17828.122  76875.365
           5       3  23.810  0.407  13583.849  38187.542
2026-07-31 0       2   0.000 -0.000  34798.628  69739.150
           2       3  25.397  0.401  40184.245  56525.621
           3       5  38.095  0.504  18512.124  70561.741
           4       7  50.794  0.547  17712.596  72083.018


## 6. Validation protocol — read before trainingThere are **53 windows but only 10 independent blocks**. Rows inside a blockare near-duplicates: same dust, minutes apart. A random split puts a window'sown twin in the test set and will report a fake MAE near zero.Two splits only, both reported in the paper:1. **Leave-one-block-out** — hold out one dust level entirely.2. **Leave-one-session-out** — train on 07-28, test on 07-31. This is the   real generalisation test and the number a reviewer will look for.**Label uncertainty floor.** The clean-clean ratio itself scatters ~0.5-1%within a session and ~2-4% between days, so a loss label is good to roughly+/-2 percentage points. Any model reporting MAE below ~0.02 is fitting noise,not soiling.

In [ ]:
def leave_one_session_out(win, sess_test="2026-07-31"):
    tr = win[win.session != sess_test]
    te = win[win.session == sess_test]
    return tr, te

FEATURES = ["vB_mean","vB_std","iB_mean","iB_std","pB_mean","lux_mean","lux_std",
            "tB_mean","tB_rise","iB_per_klux","pB_per_klux","hour_sin","hour_cos"]

tr, te = leave_one_session_out(win)
print(f"Train: {len(tr)} windows / {tr.level.nunique()} blocks")
print(f"Test:  {len(te)} windows / {te.level.nunique()} blocks")

Train: 36 windows / 6 blocks
Test:  17 windows / 4 blocks
